# Analyse complémentaire (SANS GPU) : métriques calculées après coup

**Notebook COMMUN, à lancer UNE fois par UNE personne**, quand les fichiers `results/` des 4 modèles sont réunis dans la même branche.

Il ne régénère rien : il relit les réponses déjà sauvegardées et calcule les métriques demandées par le sujet qui manquaient :

| Section | Métrique |
|---|---|
| 1 | Compilation / exécution (taux de réussite) |
| 2 | Qualité du code (complexité cyclomatique, indice de maintenabilité, score pylint) |
| 3 | Tests générés : couverture (coverage.py) et taux de tests qui passent |
| 4 | Taux d'hallucination |
| 5 | Coût par tâche et énergie : **ESTIMATIONS** (durée × prix / puissance max) |
| 6 | pass@k, vulnérabilités, énergie mesurée, mémoire CPU : lus dans les fichiers du notebook GPU `complement_gpu.ipynb` s'ils existent |

Sortie : `../resume_complementaire.csv` (+ 3 fichiers de détail).

In [ ]:
import os
REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "correctifs-gemma"   # A CHANGER : la branche qui contient les resultats des 4 modeles

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

In [ ]:
!pip install -r ../requirements.txt -q
!pip install -q radon pylint coverage

In [ ]:
import json, re, sys, subprocess, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

RESULTS = Path("../results")
MODELS = {   # memes noms que merge_results.py
    "Qwen2.5-Coder-1.5B": "qwen2.5-coder-1.5b-instruct",
    "Phi-4-mini": "phi-4-mini-instruct",
    "Gemma3-4B": "gemma-3-4b-it",
    "SmolLM3-3B": "smollm3-3b",
}

prompts_by_model, he_by_model = {}, {}
for name, slug in MODELS.items():
    p, h = RESULTS / f"benchmark_{slug}.csv", RESULTS / f"humaneval_{slug}.json"
    if p.exists(): prompts_by_model[name] = pd.read_csv(p)
    else: print("[absent]", p)
    if h.exists(): he_by_model[name] = json.load(open(h, encoding="utf-8"))
    else: print("[absent]", h)
print("Modeles avec les 15 prompts :", list(prompts_by_model))
print("Modeles avec HumanEval      :", list(he_by_model))

## Section 1 : compilation et exécution
- **compile_ok** : le code extrait (1er bloc ```) est syntaxiquement valide (`compile()`).
- **exec_ok** : le code se charge et s'exécute **sans exception** (sans les tests). Calculé pour la génération, la complétion, le refactoring et HumanEval.
- Une réponse HumanEval sans bloc de code compte comme non compilable (comme dans la notation HumanEval).

In [ ]:
HEADER = "from typing import *\nimport math\n\n"
EXEC_TASKS = {"generation_code", "completion_code", "refactoring"}

def check_compile(code):
    try:
        compile(code, "<generated>", "exec"); return True
    except Exception:
        return False

rows = []
for name, df in prompts_by_model.items():
    for _, r in df.iterrows():
        resp = str(r["response"])
        if "```" not in resp:
            continue                      # pas de code dans la reponse
        c = extract_code(resp)
        ok = check_compile(c)
        ex = None
        if r["task"] in EXEC_TASKS:
            ex = bool(ok and run_tests(HEADER + c, "pass", timeout=5)[0] == "pass")
        rows.append({"modele": name, "source": "prompts", "id": r["id"], "task": r["task"],
                     "code": c, "compile_ok": ok, "exec_ok": ex})
for name, d in he_by_model.items():
    for det in d["details"]:
        c = extract_code(det["response"])
        ok = check_compile(c)
        ex = bool(ok and run_tests(HEADER + c, "pass", timeout=5)[0] == "pass")
        rows.append({"modele": name, "source": "humaneval", "id": det["task_id"], "task": "humaneval",
                     "code": c, "compile_ok": ok, "exec_ok": ex})

df_code = pd.DataFrame(rows)
s1 = df_code.groupby("modele").agg(
    reponses_avec_code=("id", "count"),
    compile_pct=("compile_ok", lambda s: 100 * s.mean()),
    exec_pct=("exec_ok", lambda s: 100 * s.dropna().astype(bool).mean() if s.notna().any() else np.nan),
)
s1

## Section 2 : qualité du code (analyse statique)
Appliquée au code des tâches de génération, complétion, refactoring et HumanEval (code qui compile).
- **cc_moyen** : complexité cyclomatique moyenne par fonction (`radon`) : plus bas = plus simple.
- **mi_moyen** : indice de maintenabilité (`radon`, 0 à 100) : plus haut = mieux.
- **pylint_moyen** : score pylint sur 10, avec les messages de docstring manquante (C0114, C0115, C0116) et de nom indéfini (E0602) désactivés (le `HEADER` fournit `typing`). **Mêmes réglages pour tous les modèles.**

In [ ]:
from radon.complexity import cc_visit
from radon.metrics import mi_visit
from concurrent.futures import ThreadPoolExecutor

QUALITY_TASKS = {"generation_code", "completion_code", "refactoring", "humaneval"}
PYLINT_DISABLE = "C0114,C0115,C0116,E0602"

def pylint_score(code):
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "snippet.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code.rstrip() + "\n")
        try:
            out = subprocess.run([sys.executable, "-m", "pylint", path, f"--disable={PYLINT_DISABLE}",
                                  "--exit-zero", "--score=y"],
                                 capture_output=True, text=True, timeout=120).stdout
        except subprocess.TimeoutExpired:
            return np.nan
    m = re.search(r"rated at (-?[\d.]+)/10", out)
    return float(m.group(1)) if m else np.nan

def radon_metrics(code):
    try:
        blocks = cc_visit(code)
        cc = float(np.mean([b.complexity for b in blocks])) if blocks else np.nan
        mi = float(mi_visit(code, multi=True))
        return cc, mi
    except Exception:
        return np.nan, np.nan

q = df_code[df_code["compile_ok"] & df_code["task"].isin(QUALITY_TASKS)].copy()
with ThreadPoolExecutor(max_workers=4) as pool:
    q["pylint"] = list(pool.map(pylint_score, q["code"]))
q[["cc", "mi"]] = [radon_metrics(c) for c in q["code"]]

s2 = q.groupby("modele").agg(cc_moyen=("cc", "mean"), mi_moyen=("mi", "mean"), pylint_moyen=("pylint", "mean"))
q.drop(columns=["code"]).to_csv("../detail_qualite_code.csv", index=False)
s2

## Section 3 : tests générés (P14, P15) : couverture et taux de réussite
Pour chaque réponse, on extrait le code de tests, on le branche sur la **fonction de référence de l'énoncé** (le `def add` ou `def is_valid_email` redéfini par le modèle est retiré, ainsi que l'import `your_module`), puis on l'exécute sous `coverage.py` dans un processus séparé (timeout 30 s).
- **couverture_pct** : % de lignes de la fonction testée exécutées par les tests.
- **tests_reussis_pct** : % des tests générés qui passent sur la fonction de référence.

⚠ Les fonctions de P14/P15 ont 1 à 2 lignes : la couverture atteint 100 % dès que la fonction est appelée une fois. Le taux de réussite des tests est donc plus discriminant ; à écrire dans le rapport.

In [ ]:
HELPER = Path(tempfile.gettempdir()) / "run_generated_tests.py"
HELPER.write_text(r'''
import sys, os, json, ast, io, unittest
import coverage

spec = json.load(open(sys.argv[1], encoding="utf-8"))
workdir, target = spec["workdir"], spec["target"]
sys.path.insert(0, workdir)
result = {"n_tests": 0, "n_pass": 0, "n_fail": 0, "coverage_pct": None, "error": None}

try:
    tree = ast.parse(spec["test_code"])
    tree.body = [n for n in tree.body
                 if not (isinstance(n, ast.FunctionDef) and n.name == target)
                 and not (isinstance(n, ast.ImportFrom) and any(a.name == target for a in n.names))]
    code = compile(tree, "<generated_tests>", "exec")
    path = os.path.join(workdir, "target_mod.py")
    cov = coverage.Coverage(include=[path], data_file=None)
    cov.start()
    import target_mod
    ns = {"__name__": "generated_tests", target: getattr(target_mod, target)}
    exec(code, ns)
    suite = unittest.TestSuite()
    loader = unittest.TestLoader()
    for obj in list(ns.values()):
        if isinstance(obj, type) and issubclass(obj, unittest.TestCase):
            suite.addTests(loader.loadTestsFromTestCase(obj))
    if suite.countTestCases() > 0:
        res = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
        bad = len(res.failures) + len(res.errors)
        result.update(n_tests=res.testsRun, n_fail=bad, n_pass=res.testsRun - bad)
    else:
        result.update(n_tests=1, n_pass=1)   # tests de type assert : executes sans exception
    cov.stop()
    _, stmts, _, missing, _ = cov.analysis2(path)
    result["coverage_pct"] = round(100 * (len(stmts) - len(missing)) / len(stmts), 1) if stmts else None
except BaseException as e:
    result["error"] = type(e).__name__ + ": " + str(e)[:200]
    if result["n_tests"] == 0:
        result["n_tests"], result["n_fail"] = 1, 1
print("RESULT:" + json.dumps(result))
''', encoding="utf-8")

def analyse_tests(prompt, response):
    m = re.search(r"def\s+(\w+)\s*\(", prompt)
    if not m:
        return {"error": "fonction introuvable dans l'enonce"}
    func_src = prompt[prompt.index("def "):]
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, "target_mod.py"), "w", encoding="utf-8") as f:
            f.write(func_src + "\n")
        spec_path = os.path.join(d, "spec.json")
        with open(spec_path, "w", encoding="utf-8") as f:
            json.dump({"workdir": d, "target": m.group(1), "test_code": extract_code(response)}, f)
        try:
            out = subprocess.run([sys.executable, str(HELPER), spec_path], capture_output=True,
                                 text=True, timeout=30, cwd=d).stdout
        except subprocess.TimeoutExpired:
            return {"n_tests": 1, "n_pass": 0, "n_fail": 1, "coverage_pct": None, "error": "timeout"}
    lines = [l for l in out.splitlines() if l.startswith("RESULT:")]
    return json.loads(lines[-1][7:]) if lines else {"n_tests": 1, "n_pass": 0, "n_fail": 1,
                                                      "coverage_pct": None, "error": "pas de resultat"}

rows = []
for name, df in prompts_by_model.items():
    for _, r in df[df["task"] == "generation_tests"].iterrows():
        res = analyse_tests(str(r["prompt"]), str(r["response"]))
        rows.append({"modele": name, "id": r["id"], **res})
df_cov = pd.DataFrame(rows)
df_cov.to_csv("../detail_tests_generes.csv", index=False)

if len(df_cov):
    s3 = df_cov.groupby("modele").agg(couverture_pct=("coverage_pct", "mean"),
                                      tests_total=("n_tests", "sum"), tests_ok=("n_pass", "sum"))
    s3["tests_reussis_pct"] = 100 * s3["tests_ok"] / s3["tests_total"]
    s3 = s3.drop(columns=["tests_ok"])
else:
    s3 = pd.DataFrame()
df_cov

## Section 4 : taux d'hallucination
`taux = nombre de "oui" / nombre de réponses jugées (oui + non)` dans la colonne `hallucination` (notes manuelles). Les cases vides (pas de code) sont exclues.

In [ ]:
rows = []
for name, df in prompts_by_model.items():
    h = df["hallucination"].dropna().astype(str).str.strip().str.lower()
    h = h[h.isin(["oui", "non"])]
    rows.append({"modele": name, "reponses_jugees": len(h), "hallucinations": int((h == "oui").sum()),
                 "taux_hallucination_pct": 100 * (h == "oui").mean() if len(h) else np.nan})
s4 = pd.DataFrame(rows).set_index("modele")
s4

## Section 5 : coût par tâche et énergie (ESTIMATIONS)
⚠ Kaggle est gratuit et l'énergie n'est pas mesurée dans le benchmark de base : ce sont des **ordres de grandeur**.
- **cout_par_tache_usd** = latence × prix horaire de référence d'un GPU T4 (à remplacer par un tarif que vous citez dans le rapport).
- **energie_max_wh** = latence × 70 W (puissance maximale du T4) : c'est un **majorant**, pas une mesure. L'énergie réellement mesurée est en section 6.

In [ ]:
GPU_PRICE_USD_PER_HOUR = 0.35   # VALEUR INDICATIVE A REMPLACER par un tarif cite dans le rapport
T4_MAX_POWER_W = 70             # puissance maximale du NVIDIA T4

rows = []
for name, df in prompts_by_model.items():
    lat = pd.to_numeric(df["latency_s"], errors="coerce")
    out_tok = pd.to_numeric(df["output_tokens"], errors="coerce")
    rows.append({
        "modele": name,
        "cout_par_tache_usd": (lat / 3600 * GPU_PRICE_USD_PER_HOUR).mean(),
        "cout_1000_tokens_sortie_usd": lat.sum() / 3600 * GPU_PRICE_USD_PER_HOUR / out_tok.sum() * 1000,
        "energie_max_par_tache_wh": (lat / 3600 * T4_MAX_POWER_W).mean(),
    })
s5 = pd.DataFrame(rows).set_index("modele")
s5

## Section 6 : résultats du notebook GPU `complement_gpu.ipynb` (si présents)
Lit `results/passk_<modele>.json`, `results/vulnerabilites_<modele>.csv` (colonne `vuln_label` remplie) et `results/energie_<modele>.csv`. Les modèles dont les fichiers manquent restent vides.

In [ ]:
def prf1_fixed(labels):
    tp, fp, fn = (sum(l == k for l in labels) for k in ("TP", "FP", "FN"))
    p = tp / (tp + fp) if tp + fp else None
    r = tp / (tp + fn) if tp + fn else None
    if p is None or r is None:
        return p, r, None
    return p, r, (0.0 if p + r == 0 else 2 * p * r / (p + r))

rows = []
for name, slug in MODELS.items():
    d = {"modele": name}
    f = RESULTS / f"passk_{slug}.json"
    if f.exists():
        j = json.load(open(f, encoding="utf-8"))
        d.update({f"pass@{k}_%": v for k, v in j["pass_at_k"].items()})
    f = RESULTS / f"vulnerabilites_{slug}.csv"
    if f.exists():
        v = pd.read_csv(f)
        labels = [str(x).strip().upper() for x in v["vuln_label"].dropna()]
        p, r, f1 = prf1_fixed(labels)
        d.update({"vuln_precision": p, "vuln_rappel": r, "vuln_f1": f1})
    f = RESULTS / f"energie_{slug}.csv"
    if f.exists():
        e = pd.read_csv(f)
        d.update({"energie_mesuree_wh_par_tache": e["energy_wh"].mean(),
                  "energie_nette_wh_par_tache": e["energy_net_wh"].mean(),
                  "puissance_moyenne_w": e["avg_power_w"].mean(),
                  "memoire_cpu_pic_go": e["cpu_rss_peak_go"].max()})
    rows.append(d)
s6 = pd.DataFrame(rows).set_index("modele")
s6

## Résumé final

In [ ]:
parts = [s1, s2, s3, s4, s5, s6]
resume = pd.concat([p for p in parts if len(p)], axis=1)
resume = resume.reindex([m for m in MODELS if m in resume.index]).round(3)
resume.to_csv("../resume_complementaire.csv")
print("-> resume_complementaire.csv, detail_qualite_code.csv, detail_tests_generes.csv")
resume.T